# MTS Debug Test - Guerneville (No Synthetic Dates)
Minimal test to isolate 1H CSV generation issue

In [ ]:
import sys
from pathlib import Path

# Setup paths
project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

from UCB_training.UCB_train import UCB_trainer
from UCB_training.UCB_utils import data_dir

In [ ]:
# Config
BASIN = "guerneville"
path_to_csv = data_dir()
path_to_yaml = project_root / "UCB_training" / "configs" / "guerneville_mtslstm2_test.yaml"

print(f"Data dir: {path_to_csv}")
print(f"YAML: {path_to_yaml}")
print(f"YAML exists: {path_to_yaml.exists()}")

In [ ]:
# Minimal hyperparams - 1 epoch
hp_run = {
    "hidden_size": 32,
    "output_dropout": 0.4,
    "epochs": 1,
    "batch_size": 64,
    "learning_rate": {0: 0.01}
}

print(f"Hyperparams: {hp_run}")

In [ ]:
# Create trainer
trainer = UCB_trainer(
    path_to_csv_folder=path_to_csv,
    yaml_path=path_to_yaml,
    hyperparams=hp_run,
    input_features=None,
    physics_informed=False,
    physics_data_file=None,
    hourly=True,
    extend_train_period=False,
    gpu=-1,  # CPU for quick test
    is_mts=True,
    verbose=True,
    runs_parent=None,
    run_label="DEBUG_TEST",
    run_stamp=None
)

print("Trainer created successfully")

In [ ]:
# Train
print("Starting training...")
trainer.train()
print("Training complete")

In [ ]:
# Get results - 1D (should work)
print("\n=== Getting 1D results ===")
csv_1d, metrics_1d = trainer.results(period="validation", mts_trk="1D")
print(f"1D CSV: {csv_1d}")
print(f"1D CSV exists: {csv_1d.exists() if csv_1d else False}")
print(f"1D Metrics: {metrics_1d}")

In [ ]:
# Get results - 1H (this is the problematic one)
print("\n=== Getting 1H results ===")
csv_1h, metrics_1h = trainer.results(period="validation", mts_trk="1H")
print(f"1H CSV: {csv_1h}")
print(f"1H CSV exists: {csv_1h.exists() if csv_1h else False}")
print(f"1H Metrics: {metrics_1h}")

In [ ]:
# Verify CSVs
import pandas as pd

if csv_1d and csv_1d.exists():
    print("\n1D CSV head:")
    print(pd.read_csv(csv_1d).head())

if csv_1h and csv_1h.exists():
    print("\n1H CSV head:")
    print(pd.read_csv(csv_1h).head())
else:
    print("\n1H CSV NOT CREATED - check debug output above")